## Bronze to Silver: Data Cleaning and Transformation for Dimension Tables

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

catalog_name = "ecommerce"

### Category table trasformation..

In [ ]:
broze_cate = spark.table(f"{catalog_name}.broze.brz_categories") 
display(broze_cate)

In [ ]:
# Transformation: convert category code to upper case
silver_cate = broze_cate.withColumn("category_code", F.upper(F.trim(F.col("category_code"))))

# Transformation: remove space from category_name..
silver_cate = silver_cate.withColumn("category_name", F.trim(F.col("category_name")))

display(silver_cate)

In [ ]:
silver_cate.write.format("delta").mode("overwrite").option("mergeSchema", True).saveAsTable(f"{catalog_name}.silver.slv_categories")

In [ ]:
%%sql
SELECT * FROM ecommerce.silver.slv_categories;

### Brand table trasformation..

In [ ]:
broze_brand = spark.table(f"{catalog_name}.broze.brz_brands") 
display(broze_brand)

In [ ]:
# Trasnformation: remove special chnarater from brand code..
silver_brand = broze_brand.withColumn("brand_code", F.regexp_replace(F.col("brand_code"), r'[^A-Za-z0-9]', ""))

# Trasnformation: remove space from brand name
silver_brand = silver_brand.withColumn("brand_name", F.trim(F.col("brand_name")))

display(silver_brand)

In [ ]:
# Get unoque category code from brand...
silver_brand.select("category_code").distinct().show()

In [ ]:
# Anomalies dictionary
cate_code_ataomaly_dict = {
    "GROCERY": "GRCY",
    "BOOKS":"BKS",
    "TOYS":"TOY"
}

silver_brand = silver_brand.replace(to_replace=cate_code_ataomaly_dict, subset=["category_code"])

display(silver_brand)

In [ ]:
silver_brand.write.format("delta").mode("overwrite").option("mergeSchema", True).saveAsTable(f"{catalog_name}.silver.slv_brands")

In [ ]:
%%sql
SELECT * FROM ecommerce.silver.slv_brands;

### Date table trasformation..

In [ ]:
broze_date = spark.table(f"{catalog_name}.broze.brz_date") 

display(broze_date)

In [ ]:
# Transformation: convert date to yyyy-MM-dd format..
silver_calender = broze_date.withColumn("date", F.to_date(F.col("date"), "dd-MM-yyyy"))

# Transformation: convert negetive week of year to positive..
silver_calender = silver_calender.withColumn("week_of_year", F.abs(F.col("week_of_year")))

# Transformation: convert day name format ..
silver_calender = silver_calender.withColumn("day_name", F.initcap(F.col("day_name")))

display(silver_calender)

In [ ]:
# Check duplicate date..
silver_calender.groupBy("date").count().filter("count > 1").show()

# drop duplicate..
silver_calender = silver_calender.dropDuplicates(["date"])

display(silver_calender)

In [ ]:
silver_calender.write.format("delta").mode("overwrite").option("mergeSchema", True).saveAsTable(f"{catalog_name}.silver.slv_calender")

In [ ]:
%%sql
SELECT * FROM ecommerce.silver.slv_calender;

### Customer table trasformation..

In [ ]:
broze_customer = spark.table(f"{catalog_name}.broze.brz_customers") 

display(broze_customer)

In [ ]:
# Transformation: check blank customer_id, if found then drop..
null_count = broze_customer.filter(F.col("customer_id").isNull()).count()
print(f"Row count of null values: {null_count}")

silver_customer = broze_customer.dropna(subset=["customer_id"])

# Get row count
row_count = silver_customer.count()
print(f"Row count after droping null values: {row_count}")

# Transformation: Check duplicate in customer_id
silver_customer.groupBy("customer_id").count().filter("count > 1").show()

display(silver_customer)

In [ ]:
# Transformation: check phone no null value, if found the drop
null_count = silver_customer.filter(F.col("phone").isNull()).count()
print(f"Total phone no nul value: {null_count}")

silver_customer = silver_customer.dropna(subset=["phone"])

display(silver_customer.count())

In [ ]:
silver_customer.write.format("delta").mode("overwrite").option("mergeSchema", True).saveAsTable(f"{catalog_name}.silver.slv_customers")

In [ ]:
%%sql
SELECT * FROM ecommerce.silver.slv_customers LIMIT 100;

### Product table trasformation..

In [ ]:
broze_product = spark.table(f"{catalog_name}.broze.brz_products") 

display(broze_product)

In [ ]:
# Transfrmation: check null value in product_id, if found then drop
null_count = broze_product.filter(F.col("product_id").isNull()).count()
print(f"Total null value in product_id: {null_count}")

# Transformation: check duplicate product id..
broze_product.groupBy("product_id").count().filter("count > 1").show()

silver_product = broze_product.dropna(subset=["product_id"])

# Trnasformation: covert brnad & category ode to upper case,,
silver_product = silver_product.withColumn("brand_code", F.upper(F.col("brand_code"))) \
                                .withColumn("category_code", F.upper(F.col("category_code")))

# Transformation: remove g from weight_grams column and convert to float
silver_product = silver_product.withColumn("weight_grams", F.regexp_replace(F.col("weight_grams"), "g", "").cast("float"))

# Transformation: replace ',' with '.' from length_cm column and convert to float
silver_product = silver_product.withColumn( "length_cm", 
                            F.round(F.regexp_replace(F.col("length_cm"), ",", ".").cast("float"), 1))

# Transformation: take one number after . on width & height
silver_product = silver_product.withColumn("width_cm", F.round(F.col("width_cm"), 1)) \
                                .withColumn("height_cm", F.round(F.col("height_cm"), 1))


# Transformation: convert negaetive rating count to posistive..
silver_product = silver_product.withColumn("rating_count", F.abs(F.col("rating_count")))
               

display(silver_product)

In [ ]:
#  Get unique materials..
silver_product.select("material").distinct().show()

# check ing spelling mistake..
silver_product = silver_product.withColumn("material",
                            F.when(F.col("material") == "Coton", "Cotton")
                             .when(F.col("material") == "Ruber", "Rubber")
                             .when(F.col("material") == "Alumium", "Aluminum")
                             .otherwise(F.col("material"))
                          )

display(silver_product)

In [ ]:
silver_product.printSchema()

In [ ]:
silver_product.write.format("delta").mode("overwrite").option("mergeSchema", True).saveAsTable(f"{catalog_name}.silver.slv_products")

In [ ]:
%%sql
SELECT * FROM ecommerce.silver.slv_products;